# Produce AAE funtional area aggregations

This notebook produces aggregations for A&E for a given NHP Demand and Capacity model scenario. Please note the following ESSENTIAL requirements for running this notebook:

- The scenario must have been run with full model results. Provide the path to the full model results in the notebook widget at the top of the notebook, in the format `full-model-results/vx.x/PROVIDER/SCENARIO_NAME/SCENARIO_RUNTIME/`
- A .env file with the correct environment variables
- Run the `generate_token.ps1` file in your local machine Terminal _before_ running this notebook, which generates SAS tokens valid for 24 hours and sets them as Databricks secrets

## Setup

In [0]:
%cd ..
%pip install -e .
import pyspark.sql.functions as F
import uuid
from nhp.functional_areas.helpers import load_env_vars, extract_model_run_details, load_op_aae_data, validate_result_path
from datetime import datetime 
from databricks.connect import DatabricksSession
from databricks.sdk import WorkspaceClient

spark = DatabricksSession.builder.getOrCreate()
w = WorkspaceClient()
dbutils = w.dbutils

dbutils.widgets.text("capacity_model_version", "dev", "Capacity Model version")
dbutils.widgets.text("path_to_full_model_results", "", "Path to full model results")

mapping_runtime = datetime.now().strftime(format="%Y%m%d-%H%M%S")

In [0]:
path_to_full_model_results = dbutils.widgets.get('path_to_full_model_results')

db_path_to_full_model_results = str(validate_result_path(path_to_full_model_results))

env_vars = load_env_vars()

demand_model_version, fyear, provider, scenario_name, scenario_runtime = extract_model_run_details(path_to_full_model_results)

## Load data

In [0]:
aae_original = load_op_aae_data(demand_model_version, "aae", fyear, provider)
aae_model_results = spark.read.parquet(db_path_to_full_model_results + "aae")

## Produce aggregations

In [0]:
def create_aae_groupings(df):
    df = df.withColumn(
        "grouping",
        F.when(F.col("pod") == "aae_type-05", "sdec_attendances")
        .when(
            (F.col("acuity") == "immediate-resuscitation"),
            "resus_attendances",
        )
        .when(
            (F.col("is_adult") == True)
            & (F.col("pod") == "aae_type-01")
            & (F.col("acuity").isin("standard", "non-urgent", "urgent")),
            "adult_minor_attendances",
        )
        .when(
            (F.col("is_adult") == True) & (F.col("pod") == "aae_type-01") & (F.col("acuity") == "very-urgent"),
            "adult_major_attendances",
        )

        .when(
            (F.col("is_adult") == False) & (F.col("pod") == "aae_type-01")
            & (F.col("acuity").isin("standard", "non-urgent", "urgent")),
            "child_minor_attendances",
        )
        .when(
            (F.col("is_adult") == False) & (F.col("pod") == "aae_type-01") & (F.col("acuity") == "very-urgent"),
            "child_major_attendances",
        )
        .when(
            (F.col("is_adult") == False) & (F.col("pod") == "aae_type-02"),
            "child_type-02",
        )
        .when(
            (F.col("is_adult") == True) & (F.col("pod") == "aae_type-02"),
            "adult_type-02",
        )
        .when(
            (F.col("is_adult") == False),
            "child_unknown",
        )
        .when(
            (F.col("is_adult") == True),
            "adult_unknown",
        )
    )
    return df


aae_with_groups = create_aae_groupings(aae_original)

In [0]:
groupings_per_run = (
    aae_with_groups.drop("model_run", "arrivals")
    .join(aae_model_results, on="rn", how="left")
    .groupBy("model_run", "grouping")
    .agg(
        F.sum("arrivals").alias("arrivals")
    )
)

In [0]:
sdec_converted = spark.read.parquet(db_path_to_full_model_results + "sdec_conversion").withColumn("grouping", F.lit("sdec_attendances"))
sdec_groupings_per_run = sdec_converted.groupBy("model_run", "grouping").agg(
        F.sum("arrivals").alias("arrivals")
    )
final_groupings_per_run = groupings_per_run.unionByName(sdec_groupings_per_run).groupBy("model_run", "grouping").agg(
        F.sum("arrivals").alias("arrivals")
    )

In [0]:
## Add baseline and any missing groupings

def calculate_baseline(baseline_grouped):
    baseline_counts = (baseline_grouped.groupBy("model_run", "grouping")
        .agg(
            F.sum("arrivals").alias("arrivals")
        )
    )
    return baseline_counts

baseline_counts = calculate_baseline(aae_with_groups)

final_groupings_per_run_with_baseline = groupings_per_run.unionByName(baseline_counts)

def add_missing_groupings(final_groupings_per_run_with_baseline, required_groupings: list):
    model_runs = final_groupings_per_run_with_baseline.select("model_run").distinct()
    required_df = spark.createDataFrame(
        [(g,) for g in required_groupings],
        ["grouping"]
    )
    expected = model_runs.crossJoin(required_df)
    completed_required = (
        expected
        .join(
            final_groupings_per_run_with_baseline,
            on=["model_run", "grouping"],
            how="left"
        )
        .withColumn("arrivals", F.coalesce(F.col("arrivals"), F.lit(0)))
    )
    non_required = final_groupings_per_run_with_baseline.filter(~F.col("grouping").isin(required_groupings))
    final_df = non_required.unionByName(completed_required)
    return final_df

required_aae_groupings = ["adult_minor_attendances", "adult_major_attendances", "child_minor_attendances", "child_major_attendances", "resus_attendances", "sdec_attendances", "adult_unknown", "child_unknown"]
final_df = add_missing_groupings(final_groupings_per_run_with_baseline, required_aae_groupings)

In [0]:
summary = (
    final_groupings_per_run_with_baseline
    .groupBy("grouping")
    .agg(
        F.mean("arrivals").alias("mean"),
        F.expr("percentile_approx(arrivals, 0.10)").alias("p10"),
        F.expr("percentile_approx(arrivals, 0.90)").alias("p90"),
    )
)

## QA check

In [0]:

def qa_aae_results(aae_model_results, final_groupings_per_run_with_baseline):
    default_results = (
        aae_model_results.groupBy("model_run")
        .agg(F.sum("arrivals").alias("arrivals"))
        .agg(F.mean("arrivals").alias("mean_arrivals"))
        .collect()[0][0]
    )
    grouped_results = (
        final_groupings_per_run_with_baseline.filter(F.col("model_run") != 0)
        .groupBy("model_run")
        .agg(F.sum("arrivals").alias("arrivals"))
        .agg(F.mean("arrivals").alias("mean_arrivals"))
        .collect()[0][0]
    )
    assert float(default_results) == float(grouped_results)

qa_aae_results(aae_model_results, final_groupings_per_run_with_baseline)

## Upload results

In [0]:
from azure.storage.blob import ContainerClient

storage_token = dbutils.secrets.get(secret_scope, storage_key)
storage_url = dbutils.secrets.get(secret_scope, "url")
container_client = ContainerClient.from_container_url(f"{storage_url}?{storage_token}")
storage_guid = str(uuid.uuid4())


In [0]:
metadata = {
    'PartitionKey': dbutils.widgets.get('capacity_model_version'),
    'RowKey': storage_guid,
    'app_version': demand_model_version,
    'scenario_name': scenario_name,
    'scenario_runtime': scenario_runtime,
    'dataset': provider,
    'mapping_runtime': mapping_runtime,
    'path_to_full_results_dir': path_to_full_model_results
}

In [0]:
container_client.upload_blob(
    f"functional-aggregations/{dbutils.widgets.get('capacity_model_version')}/{storage_guid}/aae.parquet",
    final_df.toPandas().set_index("model_run").sort_index().to_parquet(),
    metadata=metadata,
    overwrite=True,
)
container_client.upload_blob(
    f"functional-aggregations/{dbutils.widgets.get('capacity_model_version')}/{storage_guid}/aae_summary.parquet",
    summary.toPandas().to_parquet(),
    metadata=metadata,
    overwrite=True,
)

## Add details to Azure Table Storage

In [0]:
from azure.data.tables import TableClient
from azure.core.credentials import AzureSasCredential

table_token = dbutils.secrets.get(secret_scope, table_key).strip()
table_endpoint = f"https://{account_name}.table.core.windows.net"

table_client = TableClient(
    endpoint=table_endpoint,
    table_name=table_name,
    credential=AzureSasCredential(table_token),
)

table_client.create_entity(entity=metadata)